#### This notebook implements a decoder-only transformer trained from scratch on combined data from Wikitext 103 and bookcorpus(v1) dataset.
#### Authored by Parth Jade BT2024034.

In [ ]:
## librarires below has been imported for file handling, regex, math, randomness, and garbage collection 
import os
import re
import math
import random
import gc
# deep learning core libraries (pytorch)
import torch
import torch.nn as nn
from torch.nn import functional as F

# huggingface fast tokenzier for GPT-2 (used for encoding the dataset before training)
from transformers import GPT2TokenizerFast

/home/usera19/miniconda3/envs/parthj/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True" #This allows CUDA to grow memory allocations instead of pre-reserving (reduces OOM errors)

os.environ["TOKENIZERS_PARALLELISM"] = "false" # disables tokenizer parallelism warnings 

In [ ]:

# below code checks the gpu is available along with version and vram
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Ti
CUDA version: 12.1
VRAM: 16.7 GB


In [ ]:
""""
Below class  is used to preprocess wikitext-103 dataset before tokenization and training them.
Removes WikiText artifacts:
  - @-@, @,@, @.@  : punctuation joiners which corrupts GPT-2 tokenization if used
  - <unk>           : OOV placeholders can add noise during training.
  - =+ ... =+       : section heading markers the model shouldn't learn


"""


class TextCleaner:

    def __init__(self, filepath):
        self.filepath = filepath
        self.text = None

    def load(self):
        with open(self.filepath, "r", encoding="utf-8") as f:
            self.text = f.read()
        print(f"[INFO] Loaded dataset : {len(self.text):,} chars")
        return self

    def fix_artifacts(self):
        #below replace wikiText tokenization joiners with their actual characters
        self.text = self.text.replace("@-@", "-")
        self.text = self.text.replace("@,@", ",")
        self.text = self.text.replace("@.@", ".")
        self.text = self.text.replace("<unk>", "")
        print("[INFO] Artifacts fixed")
        return self

    def mark_section_boundaries(self):
        # wikiText uses repeated "=" signs to denote headings for each section.
         # It is replaced with double newlines to preserve paragraph structure
        self.text = re.sub(r"=+.*?=+", "\n\n", self.text)
        print("[INFO] Section boundaries marked")
        return self

    def clean_lines(self):
        lines = self.text.split("\n")
        cleaned = [line.strip() for line in lines if line.strip()]
        self.text = "\n".join(cleaned)
        print("[INFO] Lines cleaned")
        return self

    def normalize_whitespace(self):
        self.text = re.sub(r"[ \t]+", " ", self.text) # collapse multiple spaces/tabs into a single space
        self.text = re.sub(r"\n{3,}", "\n\n", self.text) #3+ consecutive newlines is collapsed  down to 2 (paragraph break)
        self.text = self.text.strip()
        print("[INFO] Whitespace normalized")
        return self

    def run(self):
        self.load()
        self.fix_artifacts()
        self.mark_section_boundaries()
        self.clean_lines()
        self.normalize_whitespace()
        return self

    def report(self, n=300):
        print(f"\n[INFO] Final text size : {len(self.text):,} chars")
        print(f"[INFO] Sample          :\n{self.text[:n]}")
        return self

    def get_text(self):
        if self.text is None:
            raise RuntimeError("TextCleaner: call .run() before .get_text()")
        return self.text

In [ ]:
"""
Below class is designed to handle large datasets without memory issues. 

The main idea is the encode_to_file() function.Instead of storing  all
tokens in a huge Python list (which can take several GBs of RAM for large
datasets which can cause OOM error),it processes the text in chunks and writes tokens directly
to a binary file.This makes it much more memory-efficient and scalable.


  
GPT2TokenizerFast is used over GPT2Tokenizer as it  runs on a Rust backend, making it 5 to 10x faster for our large corpra.
"""

class TokenizerWrapper:
    def __init__(self, model_name="gpt2"):
        self.model_name  = model_name
        self.tokenizer   = None
        self.vocab_size  = None

    def load(self):
        self.tokenizer = GPT2TokenizerFast.from_pretrained(self.model_name)
        self.tokenizer.model_max_length = 10**9
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.vocab_size = self.tokenizer.vocab_size
        print(f"[INFO] Tokenizer loaded  : {self.model_name}")
        print(f"[INFO] Vocab size        : {self.vocab_size:,}")
        print(f"[INFO] EOS token ID      : {self.tokenizer.eos_token_id}")
        return self

    def encode(self, s):
        return self.tokenizer.encode(s, add_special_tokens=False)

    def decode(self, tokens):
        return self.tokenizer.decode(tokens, skip_special_tokens=True)

    def encode_to_file(self, text, out_path, chunk_chars=100_000):
        import numpy as np
        import struct

        eos_id = self.tokenizer.eos_token_id
        total_chars = len(text)
        total_tokens = 0
        with open(out_path, "wb") as f:
            pos = 0
            while pos < total_chars:
                end = min(pos + chunk_chars, total_chars)
                chunk = text[pos:end]

                if end < total_chars:
                    last_space = chunk.rfind(" ")
                    if last_space != -1:
                        chunk = chunk[:last_space]
                        end = pos + last_space
                encoded = self.tokenizer.encode(chunk, add_special_tokens=False)
                encoded.append(eos_id)   #one EOS per chunk (document boundary)
                # written in numpy array (int32)- 4 bytes per token
                arr = np.array(encoded, dtype=np.int32)
                f.write(arr.tobytes())
                total_tokens += len(encoded)
                pos = end
                pct = min(100, int(pos / total_chars * 100))
                print(f"\r[INFO] Encoding... {pct:3d}%  ({total_tokens:,} tokens)", end="", flush=True)
        print()
        print(f"[INFO] Total tokens written: {total_tokens:,}")
        print(f"[INFO] File size: {os.path.getsize(out_path) / 1e9:.2f} GB")
        return total_tokens

    def sanity_check(self):
        #below code does a quick round-trip test to confirm encode/decode work correctly
        test   = "hello world <|endoftext|> next document"
        tokens = self.encode(test)
        decoded = self.decode(tokens)
        print(f"[INFO] Sanity check")
        print(f"       input   : {test}")
        print(f"       tokens  : {tokens}")
        print(f"       decoded : {decoded}")
        assert self.tokenizer.eos_token_id == 50256, \
            "EOS token ID mismatch - expected 50256"
        print(f"       EOS ID  : {self.tokenizer.eos_token_id} ok")
        return self

In [ ]:
class DataManager:
    # this class manages loading and splitting of tokenized dataset into train and validation sets.
     #It is splitted this way: 90% of tokens for training, 10% for validation.
     
    def __init__(self, data=None):
        self.data = data
        self.train_data = None
        self.val_data = None

    def load_from_bin(self, path, total_tokens):
        import numpy as np
        print(f"[INFO] Loading tokens from {path}...")
        arr = np.fromfile(path, dtype=np.int32)
        assert len(arr) == total_tokens, f"Expected {total_tokens}, got {len(arr)}"
        self.data = torch.from_numpy(arr).to(torch.long)  
        print(f"[INFO] Loaded {len(self.data):,} tokens")
        print(f"[INFO] Tensor size: {self.data.element_size() * len(self.data) / 1e9:.2f} GB")
        return self

    def split(self, ratio=0.9):
        n = int(ratio * len(self.data))
        self.train_data = self.data[:n]
        self.val_data   = self.data[n:]

        self.train_data = self.train_data.contiguous()
        self.val_data   = self.val_data.contiguous()
        del self.data
        self.data = None
        gc.collect()

        print(f"Train tokens : {len(self.train_data):,}")
        print(f"Val tokens   : {len(self.val_data):,}")
        print(f"Train size   : {self.train_data.element_size() * len(self.train_data) / 1e9:.2f} GB")
        print(f"Val size     : {self.val_data.element_size() * len(self.val_data) / 1e9:.2f} GB")
        return self

    def get_data(self):
        return self.train_data, self.val_data

In [ ]:
class Config:
# This class stores hyperparameter for training the transformer.
    def __init__(self):
        self.batch_size    = 16       
        self.block_size    = 256     #context window length in tokens
        self.max_iters     = 50000
        self.learning_rate =1e-4
        self.n_embd        = 512       #embedding / hidden dimension
        self.n_head        = 8       # number of attention heads
        self.n_layer       = 8        # number of transformer decoder blocks  
        self.grad_accum    = 8        
        self.dropout       = 0.1     # dropout rate  applied in feedforward and attention  layers to prevent overfitting by randomly zeroing activations during training
        self.eval_interval= 500       
        self.eval_iters    = 100
        self.warmup_steps  = 1000
        self.patience      = 10
        self.max_grad_norm = 1.0      # gradient clipping threshold 
        self.device        = "cuda" if torch.cuda.is_available() else "cpu"  #check if cuda is available otherwise switch to cpu

    def summary(self):
        print("\n===== CONFIG =====")
        for k, v in vars(self).items():
            print(f"{k:15}: {v}")
        return self

In [ ]:
class HardwareManager:
# below class is used to manage gpu memory during the training process. It is used to diagnose OOM errors.
    def __init__(self):
        pass

    def gpu_usage(self):
        if torch.cuda.is_available():
            allocated=torch.cuda.memory_allocated() / 1e9
            reserved=torch.cuda.memory_reserved() / 1e9
            total=torch.cuda.get_device_properties(0).total_memory / 1e9
            free= total - allocated
            print(f"GPU allocated : {allocated:.2f} GB")
            print(f"GPU reserved  : {reserved:.2f} GB")
            print(f"GPU free      : {free:.2f} GB")
        else:
            print("Running on CPU")

    def info(self):
        print("\n===== HARDWARE INFO =====")
        if torch.cuda.is_available():
            print("GPU        :", torch.cuda.get_device_name(0))
            print("CUDA ver   :", torch.version.cuda)
            print("Total VRAM :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
        self.gpu_usage()
        return self
#force garbage collection and release PyTorch's CUDA memory cache
    def reset_gpu(self):
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print("GPU reset complete")
        self.gpu_usage()
        return self

In [ ]:
"""
generates random  mini batches of token sequences for training.
Each batch contains input tokens (x) and next-token targets (y).
avoids extra memory overhead using slicing and speeds up GPU transfer.
"""

class BatchLoader:
    def __init__(self, train_data, val_data, block_size, batch_size, device):
        self.train_data = train_data
        self.val_data=val_data
        self.block_size=block_size
        self.batch_size=batch_size
        self.device=device

    def get_batch(self, split):
        src = self.train_data if split == "train" else self.val_data
        ix = torch.randint(0, len(src) - self.block_size - 1, (self.batch_size,))

        x = torch.stack([src[i : i + self.block_size] for i in ix])
        y = torch.stack([src[i + 1 : i + 1 + self.block_size] for i in ix])

        if self.device == "cuda":
            x = x.pin_memory().to(self.device, non_blocking=True)
            y = y.pin_memory().to(self.device, non_blocking=True)
        else:
            x = x.to(self.device)
            y = y.to(self.device)
        return x, y

In [ ]:
class Head(nn.Module):

    """
    Single causal self-attention head — tokens can only attend to past positions.
    """
    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.key= nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value=nn.Linear(n_embd, head_size, bias=False)
        #stored once as a buffer — auto-moves to GPU, cropped to T at runtime
        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size)).bool()
        )
        self.attn_dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T, C = x.shape
        k=self.key(x)
        q=self.query(x)
        #scale by 1/√head_size to prevent softmax saturation
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        #crop mask to T at runtime
        wei = wei.masked_fill(~self.tril[:T, :T], float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.attn_dropout(wei)
        v=self.value(x)
        return wei @ v

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Runs n_head attention heads in parallel, then concatenates and projects.
    """
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size =n_embd // n_head
        self.heads = nn.ModuleList([
            Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)
        ])

        # Linear projection to mix information across heads after concatenation
        self.proj=nn.Linear(n_embd, n_embd)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.resid_dropout(self.proj(out))

In [ ]:
class FeedForward(nn.Module):
    """
    Below code implements position-wise feed-forward network which is applied after each attention sublayer.
      
      GELU activation is used instead of ReLU, consistent
    with GPT-2's architecture — GELU is smoother and empirically performs
    better on language tasks.
    """
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
"""
Below code implements one full transformer decoder block.

Uses pre-norm (LayerNorm before each sublayer) rather than the post-norm.
Residual connections around both sublayers allow gradients to flow
directly to early layers, addressing vanishing gradients in deep stacks.

"""

class Block(nn.Module):

    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1=nn.LayerNorm(n_embd)
        self.sa= MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ln2= nn.LayerNorm(n_embd)
        self.ff=FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [ ]:
class GPT(nn.Module):
    """
    Below code implements Decoder only language model.
    
    Summary of the Architecture:
      1) Token embedding+positional embedding (learned, not sinusoidal)
      2)  Final LayerNorm before language model head
      3)n_layer transformer decoder blocks

    generate function supports temperature scaling, top-k filtering, and nucleus
    (top-p) sampling for controlling output diversity at inference time.
    """
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout, device):
        super().__init__()
        self.device = device
        self.block_size = block_size
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)
        ])
        self.ln_f= nn.LayerNorm(n_embd) 
        self.lm_head= nn.Linear(n_embd,vocab_size,bias=False)
        self.lm_head.weight = self.token_embedding_table.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.token_embedding_table(idx)
        pos = self.position_embedding_table(torch.arange(T, device=self.device))
        x   = self.blocks(tok + pos)
        x   = self.ln_f(x)
        logits = self.lm_head(x)


        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(B * T, -1),
                targets.view(B * T)
            )
        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=0.9, top_k=50, top_p=0.9):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, -1, None]] = -float("inf")

            probs = F.softmax(logits, dim=-1)

            if top_p < 1.0:
                sorted_probs, sorted_idx = torch.sort(probs, descending=True)
                cumulative = torch.cumsum(sorted_probs, dim=-1)
                mask = cumulative - sorted_probs > top_p
                sorted_probs[mask] = 0
                probs = torch.zeros_like(probs).scatter_(1, sorted_idx, sorted_probs)
                probs = probs / probs.sum(dim=-1, keepdim=True)

            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [15]:
class ModelManager:
    def __init__(self, config, vocab_size):
        self.config = config
        self.vocab_size = vocab_size
        self.model = None

    def build(self):
        self.model = GPT(
            self.vocab_size,
            self.config.n_embd,
            self.config.n_head,
            self.config.n_layer,
            self.config.block_size,
            self.config.dropout,
            self.config.device
        ).to(self.config.device)

        self.model = torch.compile(self.model)

        total_params = sum(p.numel() for p in self.model.parameters())

        print(f"Model parameters : {total_params:,}")
        print(f"Estimated size   : {total_params * 4 / 1e6:.1f} MB (fp32)")

        return self

    def get_model(self):
        return self.model

In [ ]:
class Trainer:
    """
     Handles training with mixed precision,grad accumulation,clipping, LR scheduling,and early stopping.
    """
    def __init__(self, model, config, batch_loader):
        self.model = model
        self.config = config
        self.batch_loader = batch_loader

        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config.learning_rate
        )

        self.scaler = torch.amp.GradScaler("cuda", enabled=(config.device == "cuda"))

        self.best_val_loss = float("inf")
        self.no_improve_steps = 0

    def get_lr(self, step):
        if step < self.config.warmup_steps:
            return self.config.learning_rate * step / self.config.warmup_steps
        progress = (step - self.config.warmup_steps) / (self.config.max_iters - self.config.warmup_steps)
        return 0.5 * self.config.learning_rate * (1 + math.cos(math.pi * progress))

    @torch.no_grad()
    def evaluate(self):
        self.model.eval()   
        losses = {}

        for split in ["train", "val"]:
            split_losses = torch.zeros(self.config.eval_iters)

            for k in range(self.config.eval_iters):
                x, y = self.batch_loader.get_batch(split)

                with torch.amp.autocast("cuda", enabled=(self.config.device == "cuda")):
                    _, loss = self.model(x, y)

                split_losses[k] = loss.item()
                del x, y, loss 

            losses[split] = split_losses.mean().item()

        torch.cuda.empty_cache()
        self.model.train()
        return losses

    def train(self):
        print("\n===== TRAINING START =====")

        for iter in range(self.config.max_iters):
            lr = self.get_lr(iter)
            for param_group in self.optimizer.param_groups:
                param_group['lr'] = lr

            total_loss = 0.0

            for _ in range(self.config.grad_accum):
                x, y = self.batch_loader.get_batch("train")

                with torch.amp.autocast("cuda", enabled=(self.config.device == "cuda")):
                    _, loss = self.model(x, y)
                    loss = loss / self.config.grad_accum

                self.scaler.scale(loss).backward()
                total_loss += loss.item()
                del x, y, loss  

    
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.max_grad_norm)

            self.scaler.step(self.optimizer)
            self.scaler.update()
            self.optimizer.zero_grad(set_to_none=True)

            if iter % self.config.eval_interval == 0:
                losses = self.evaluate()

                print(f"iter {iter:6d} | "
                      f"train {losses['train']:.4f} | "
                      f"val {losses['val']:.4f} | "
                      f"lr {lr:.6f}")

                if losses["val"] < self.best_val_loss:
                    self.best_val_loss = losses["val"]
                    self.no_improve_steps = 0
                    torch.save(self.model.state_dict(), "best_model.pt")
                    print("  -> best model saved")
                else:
                    self.no_improve_steps += 1

                if self.no_improve_steps >= self.config.patience:
                    print("Early stopping triggered")
                    break

        print("===== TRAINING COMPLETE =====")

## Step 1: Merge wikitext files (run once)

In [17]:
files = [
    "wikitext-103/wiki.train.tokens",
    "wikitext-103/wiki.valid.tokens",
    "wikitext-103/wiki.test.tokens"
]

output_path = "wikitext-103/wiki.full.tokens"
total_written = 0

with open(output_path, "w", encoding="utf-8") as outfile:
    for fname in files:
        print("Reading:", fname)
        with open(fname, "r", encoding="utf-8") as infile:
            content = infile.read()
            print(f"  chars read: {len(content):,}")
            outfile.write(content + "\n")
            total_written += len(content)

print(f"\nDone. Total chars written: {total_written:,}")

Reading: wikitext-103/wiki.train.tokens
  chars read: 382,123,643
Reading: wikitext-103/wiki.valid.tokens
  chars read: 1,140,678
Reading: wikitext-103/wiki.test.tokens
  chars read: 1,279,333

Done. Total chars written: 384,543,654


## Step 2: Clean text

In [18]:
cleaner = TextCleaner("wikitext-103/wiki.full.tokens")
cleaner.run().report()
wiki_text = cleaner.get_text()

[INFO] Loaded dataset : 384,543,657 chars
[INFO] Artifacts fixed
[INFO] Section boundaries marked
[INFO] Lines cleaned
[INFO] Whitespace normalized

[INFO] Final text size : 374,219,553 chars
[INFO] Sample          :
Senjō no Valkyria 3 : Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role - playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Ja


## Step 3: Tokenize both datasets to disk (memory-safe)
Instead of loading both texts into memory and building a giant Python list,
we stream tokens directly to a binary file one chunk at a time.

In [ ]:
tok_wrapper = TokenizerWrapper("gpt2").load()
tok_wrapper.sanity_check()

print("\n[INFO] Encoding wikitext...")
wiki_tokens = tok_wrapper.encode_to_file(wiki_text, "wiki_tokens.bin")

del wiki_text, cleaner
gc.collect()

print("\n[INFO] Loading BookCorpus...")
with open("all_books_300M.txt", "r", encoding="utf-8") as f:
    book_text = f.read()
print(f"[INFO] BookCorpus: {len(book_text):,} chars")

print("[INFO] Encoding BookCorpus...")
book_tokens = tok_wrapper.encode_to_file(book_text, "book_tokens.bin")

del book_text
gc.collect()

print("\n[INFO] Merging token files...")
with open("combined_tokens.bin", "wb") as out:
    for path in ["wiki_tokens.bin", "book_tokens.bin"]:
        with open(path, "rb") as inp:
            while True:
                chunk = inp.read(64 * 1024 * 1024)  # 64MB at a time
                if not chunk:
                    break
                out.write(chunk)

total_tokens = wiki_tokens + book_tokens
vocab_size = tok_wrapper.vocab_size
print(f"[INFO] Combined tokens: {total_tokens:,}")
print(f"[INFO] File size: {os.path.getsize('combined_tokens.bin') / 1e9:.2f} GB")

os.remove("wiki_tokens.bin")
os.remove("book_tokens.bin")
del tok_wrapper
gc.collect()

[INFO] Tokenizer loaded  : gpt2
[INFO] Vocab size        : 50,257
[INFO] EOS token ID      : 50256
[INFO] Sanity check
       input   : hello world <|endoftext|> next document
       tokens  : [31373, 995, 220, 50256, 1306, 3188]
       decoded : hello world  next document
       EOS ID  : 50256 ok

[INFO] Encoding wikitext...
[INFO] Encoding... 100%  (79,235,372 tokens)
[INFO] Total tokens written: 79,235,372
[INFO] File size: 0.32 GB

[INFO] Loading BookCorpus...
[INFO] BookCorpus: 1,112,172,617 chars
[INFO] Encoding BookCorpus...
[INFO] Encoding... 100%  (277,206,971 tokens)
[INFO] Total tokens written: 277,206,971
[INFO] File size: 1.11 GB

[INFO] Merging token files...
[INFO] Combined tokens: 356,442,343
[INFO] File size: 1.43 GB


0

## Step 4: Load tokenized data and split

In [20]:
data_manager = DataManager()
data_manager.load_from_bin("combined_tokens.bin", total_tokens)
data_manager.split()
train_data, val_data = data_manager.get_data()
del data_manager
gc.collect()

[INFO] Loading tokens from combined_tokens.bin...
[INFO] Loaded 356,442,343 tokens
[INFO] Tensor size: 2.85 GB
Train tokens : 320,798,108
Val tokens   : 35,644,235
Train size   : 2.57 GB
Val size     : 0.29 GB


0

## Step 5: Setup and train

In [21]:
config = Config().summary()


===== CONFIG =====
batch_size     : 16
block_size     : 256
max_iters      : 50000
learning_rate  : 0.0001
n_embd         : 512
n_head         : 8
n_layer        : 8
grad_accum     : 8
dropout        : 0.1
eval_interval  : 500
eval_iters     : 100
warmup_steps   : 1000
patience       : 10
max_grad_norm  : 1.0
device         : cuda


In [22]:
batch_loader = BatchLoader(
    train_data,
    val_data,
    config.block_size,
    config.batch_size,
    config.device
)

In [23]:
model = ModelManager(config, vocab_size).build().get_model()

Model parameters : 51,070,464
Estimated size   : 204.3 MB (fp32)


In [ ]:
trainer = Trainer(model, config, batch_loader)
trainer.train()


===== TRAINING START =====
iter      0 | train 10.9179 | val 10.9226 | lr 0.000000
  -> best model saved
iter    500 | train 6.0132 | val 5.5285 | lr 0.000050
  -> best model saved
iter   1000 | train 5.3342 | val 4.8569 | lr 0.000100
  -> best model saved
iter   1500 | train 4.9485 | val 4.4964 | lr 0.000100
  -> best model saved
iter   2000 | train 4.7410 | val 4.3045 | lr 0.000100
  -> best model saved
iter   2500 | train 4.5901 | val 4.1676 | lr 0.000100
  -> best model saved
iter   3000 | train 4.4506 | val 4.0859 | lr 0.000100
  -> best model saved
iter   3500 | train 4.3463 | val 4.0028 | lr 0.000099
  -> best model saved
iter   4000 | train 4.2710 | val 3.9109 | lr 0.000099
  -> best model saved
iter   4500 | train 4.2091 | val 3.8848 | lr 0.000099
  -> best model saved
iter   5000 | train 4.1312 | val 3.8077 | lr 0.000098
  -> best model saved
iter   5500 | train 4.0766 | val 3.7800 | lr 0.000098
  -> best model saved
iter   6000 | train 4.0286 | val 3.7367 | lr 0.000097
  ->

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "config": vars(config),
    "vocab_size": vocab_size
}, "final_checkpoint.pt")
print("Final checkpoint saved.")